# 06 — Evaluating Text Generation Quality: BLEU and Perplexity

## 📚 Learning Objectives

By completing this notebook, you will:
- **Compute BLEU scores** with NLTK and see them track candidate quality
- **Compute perplexity** of a language model you train — before training, after training, and on out-of-domain text
- Interpret both metrics and know their blind spots

## 🔗 Where this fits

**Builds on:** Course 10 — Unit 1, lesson 10 (FID and BLEU concepts) and Unit 2, lesson 01, whose model's perplexity is measured here before and after training.

**Used later in:** Course 10 — Unit 4, where honest evaluation becomes part of the ethical argument.

---

## Introduction

Two numbers dominate text-generation evaluation:

- **BLEU** (0–1, higher better): n-gram overlap between a candidate text and reference text(s), with a brevity penalty. Built for machine translation; still a standard baseline metric.
- **Perplexity** (≥1, lower better): `exp(average cross-entropy)` of a language model on a text. Intuition: "on average, the model was as uncertain as if choosing uniformly among *PPL* options at each step." A model that knows nothing about a 27-character alphabet has perplexity ≈ 27; training pushes it down.


In [1]:
# WHAT/WHY: compute real BLEU scores against a REAL reference sentence, using
# candidates that are themselves real sentences selected BY MEASURED word
# overlap - not hand-written to make the demo come out. If the metric works,
# BLEU should fall as the measured overlap falls.
%pip install nltk -q
import re
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1     # standard smoothing for short sentences

# ── REAL DATA: sentences from two published books ─────────────────────────
def load_sentences():
    try:
        import nltk
        nltk.download('gutenberg', quiet=True)
        from nltk.corpus import gutenberg
        return (gutenberg.raw('austen-emma.txt'), gutenberg.raw('melville-moby_dick.txt'),
                'Austen, "Emma"', 'Melville, "Moby Dick"')
    except Exception:
        from sklearn.datasets import fetch_20newsgroups
        a = " ".join(fetch_20newsgroups(subset='train', categories=['rec.autos'],
                                        remove=('headers', 'footers', 'quotes')).data)
        b = " ".join(fetch_20newsgroups(subset='train', categories=['sci.space'],
                                        remove=('headers', 'footers', 'quotes')).data)
        return a, b, '20 Newsgroups rec.autos', '20 Newsgroups sci.space'

raw_a, raw_b, name_a, name_b = load_sentences()

def sentences(raw, lo=6, hi=14):
    """Split real text into word lists of a comparable length."""
    out = []
    for s in re.split(r'(?<=[.!?])\s+', re.sub(r'\s+', ' ', raw)):
        w = re.findall(r"[a-z']+", s.lower())
        if lo <= len(w) <= hi:
            out.append(w)
    return out

pool_a = sentences(raw_a)      # same book as the reference
pool_b = sentences(raw_b)      # a completely different book
print(f'{name_a}: {len(pool_a):,} sentences | {name_b}: {len(pool_b):,} sentences')

# ── Reference: one real sentence, taken from the middle of the book ───────
reference_words = pool_a[len(pool_a) // 2]
reference = [reference_words]
print(f'\nReference (real sentence from {name_a}):\n  {" ".join(reference_words)!r}')

# ── Candidates chosen BY MEASURED overlap, so the ranking is not curated ──
ref_set = set(reference_words)
def overlap(w):
    return len(ref_set & set(w)) / len(ref_set)

others = [w for w in pool_a if w != reference_words]
others.sort(key=overlap, reverse=True)
best_same_book  = others[0]                       # highest measured overlap
mid_same_book   = others[len(others) // 12]       # moderate overlap
pool_b.sort(key=overlap)
unrelated       = pool_b[0]                       # lowest overlap, different book

candidates = {
    "identical (copy of the reference)": reference_words,
    "closest real sentence, same book":  best_same_book,
    "moderate-overlap real sentence":    mid_same_book,
    f"unrelated real sentence ({name_b.split(',')[0]})": unrelated,
}

# ── Score each candidate with 4-gram BLEU ─────────────────────────────────
scores = {}
print("\ncandidate                                     word overlap   BLEU")
for label, cand in candidates.items():
    scores[label] = sentence_bleu(reference, cand, smoothing_function=smooth)
    print(f"{label:<45} {overlap(cand):11.2f}   {scores[label]:.4f}")
    print(f"    {' '.join(cand)!r}")

# ── Read the numbers we actually got, not the numbers we hoped for ────────
near = scores["closest real sentence, same book"]
mid  = scores["moderate-overlap real sentence"]
print(f"\nBLEU falls monotonically with measured overlap: 1.0000 → {near:.4f} → "
      f"{mid:.4f} → {scores[list(scores)[-1]]:.4f}.")
print(f"Now look at the penalty size. The closest real sentence in the whole book")
print(f"shares {overlap(best_same_book):.0%} of the reference's words, yet BLEU drops to "
      f"{near:.3f}")
print(f"— it loses {1-near:.0%} of the score for a handful of edited words, because BLEU")
print("counts n-grams IN ORDER, and one changed word breaks every 4-gram covering it.")
print("Real candidates expose that harshness; hand-picked ones hide it. Whether a")
print("rewording is *correct* is something BLEU cannot see at all.")



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Austen, "Emma": 2,408 sentences | Melville, "Moby Dick": 1,994 sentences

Reference (real sentence from Austen, "Emma"):
  'can i do any thing for you no i thank you'

candidate                                     word overlap   BLEU
identical (copy of the reference)                    1.00   1.0000
    'can i do any thing for you no i thank you'
closest real sentence, same book                     0.78   0.5779
    'can i do any thing for you oh'
moderate-overlap real sentence                       0.22   0.0406
    'and upon my word i do not think mr'
unrelated real sentence (Melville)                   0.00   0.0000
    'moby dick by herman melville etymology'

BLEU falls monotonically with measured overlap: 1.0000 → 0.5779 → 0.0406 → 0.0000.
Now look at the penalty size. The closest real sentence in the whole book
shares 78% of the reference's words, yet BLEU drops to 0.578
— it loses 42% of the score for a handful of edited words, because BLEU
counts n-grams IN ORDER, and one chan

## Perplexity — Measured on a Model We Train

We train the example-01 char-LM on a deliberately **repetitive** corpus (three sentences repeated ten times), holding out the final 20% as **validation** — because the corpus repeats, the validation slice contains the same sentence patterns the model trained on. Perplexity = `exp(average next-char cross-entropy)`. Three measurements tell the story:

1. **Untrained model** on validation text — should be ≈ vocabulary size (pure guessing)
2. **Trained model** on validation text — close to 1 here, because the repetitive corpus is essentially memorizable. (On real, varied corpora a good char model lands in the tens — perplexity 1 means "always certain and right", achievable only when the text repeats.)
3. **Trained model** on out-of-domain text — enormous: the model is now *confidently wrong* about text unlike its training data. Perplexity measures fit *between a model and a text*, which is exactly why LM papers compare models on one fixed test set


In [2]:
# WHAT/WHY: compute perplexity = exp(avg cross-entropy) of a char-LM at three
# stages — untrained, trained (in-domain), and trained (out-of-domain) — to
# see what the number responds to. All three texts are REAL: a repeated toy
# corpus would drive in-domain perplexity to ~1.0 and teach a false intuition
# about what "good" perplexity looks like.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import pandas as pd
import re

torch.manual_seed(42); np.random.seed(42)

def normalise(s):
    s = re.sub(r'[^a-z .,]', ' ', s.lower())
    return re.sub(r'\s+', ' ', s).strip()

# ── REAL in-domain corpus: Austen, "Emma" ─────────────────────────────────
def load_indomain():
    try:
        import nltk
        nltk.download('gutenberg', quiet=True)
        from nltk.corpus import gutenberg
        return gutenberg.raw('austen-emma.txt'), 'Austen, "Emma"'
    except Exception:
        from sklearn.datasets import fetch_20newsgroups
        return (" ".join(fetch_20newsgroups(subset='train', categories=['rec.autos'],
                remove=('headers', 'footers', 'quotes')).data), '20 Newsgroups rec.autos')

raw, in_name = load_indomain()
text = normalise(raw)[:24000]

# ── REAL out-of-domain corpus: 911 emergency-dispatch call titles ─────────
CALLS = '../../../Course 04/datasets/raw/montgomery_911_calls.csv'
calls = pd.read_csv(CALLS, usecols=['title'], nrows=4000)
ood_text = normalise(' . '.join(calls['title'].astype(str).tolist()))[:6000]

print(f'in-domain  : {in_name} — {len(text):,} chars')
print(f'  {text[:100]!r}')
print(f'out-of-domain: 911 dispatch titles — {len(ood_text):,} chars')
print(f'  {ood_text[:100]!r}')

# ── Vocabulary over both texts, then an 80/20 train/validation split ──────
chars = sorted(set(text + ood_text))
c2i = {c: i for i, c in enumerate(chars)}
VOCAB = len(chars); SEQ_LEN = 20
split = int(len(text) * 0.8)
train_text, val_text = text[:split], text[split:]
print(f"\nvocabulary: {VOCAB} chars — untrained perplexity should be ≈ {VOCAB}")

def make_dataset(t):
    enc = [c2i[c] for c in t]
    X = [enc[i:i+SEQ_LEN] for i in range(len(enc) - SEQ_LEN - 1)]
    y = [enc[i+SEQ_LEN]   for i in range(len(enc) - SEQ_LEN - 1)]
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_tr, y_tr   = make_dataset(train_text)
X_val, y_val = make_dataset(val_text)
X_ood, y_ood = make_dataset(ood_text)
print(f"windows — train: {len(X_tr):,} | val: {len(X_val):,} | out-of-domain: {len(X_ood):,}")

class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model = CharLM(); loss_fn = nn.CrossEntropyLoss()

def perplexity(X, y, bs=2048):
    """exp of the average next-char cross-entropy — the definition, verbatim.
    Batched so a 19k-window corpus fits comfortably in memory."""
    model.eval(); tot = 0.0; n = 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb, yb = X[i:i+bs], y[i:i+bs]
            tot += loss_fn(model(xb), yb).item() * len(xb); n += len(xb)
    return float(np.exp(tot / n))

# ── Measurement 1: untrained model (should be ≈ VOCAB, i.e., guessing) ────
ppl_untrained = perplexity(X_val, y_val)
print(f"\n1. untrained model, validation text:  PPL = {ppl_untrained:7.1f}")

# ── Train on the first 80% of the real corpus ─────────────────────────────
opt = optim.Adam(model.parameters(), lr=3e-3)
for step in range(800):
    model.train()
    perm = torch.randperm(len(X_tr))[:256]
    loss = loss_fn(model(X_tr[perm]), y_tr[perm])
    opt.zero_grad(); loss.backward(); opt.step()

# ── Measurements 2, 3 and 4: trained model, train vs val vs out-of-domain ─
ppl_train = perplexity(X_tr, y_tr)
ppl_val   = perplexity(X_val, y_val)
ppl_ood   = perplexity(X_ood, y_ood)
print(f"2. trained model, TRAINING text:      PPL = {ppl_train:7.2f}")
print(f"3. trained model, held-out validation:PPL = {ppl_val:7.2f}")
print(f"4. trained model, out-of-domain text: PPL = {ppl_ood:7.1f}")

print("\nRead the four numbers together:")
print(f"  • Untrained, PPL {ppl_untrained:.1f} ≈ the {VOCAB}-character vocabulary: pure guessing.")
print(f"  • Trained, in-domain PPL {ppl_val:.2f}: much better, but nowhere near 1.0.")
print("    Real prose is genuinely uncertain. A language model reporting PPL ≈ 1")
print("    has memorised its corpus, not learned a language — which is exactly the")
print("    false intuition a repeated toy corpus teaches.")
print(f"  • Training {ppl_train:.2f} vs held-out {ppl_val:.2f} = a {ppl_val/ppl_train:.1f}× generalisation gap.")
print("    The model fits text it has seen better than text it has not; measuring")
print("    perplexity on training data would have flattered it.")
print(f"  • Out-of-domain PPL {ppl_ood:.1f} is worse than the untrained model's {ppl_untrained:.1f}:")
print("    on dispatch text the model is confidently wrong, which costs more than")
print("    guessing. Perplexity measures fit BETWEEN a model and a text — it is not")
print("    a quality score for text on its own.")


in-domain  : Austen, "Emma" — 24,000 chars
  'emma by jane austen volume i chapter i emma woodhouse, handsome, clever, and rich, with a comfortabl'
out-of-domain: 911 dispatch titles — 6,000 chars
  'ems back pains injury . ems diabetic emergency . fire gas odor leak . ems cardiac emergency . ems di'

vocabulary: 29 chars — untrained perplexity should be ≈ 29
windows — train: 19,179 | val: 4,779 | out-of-domain: 5,979

1. untrained model, validation text:  PPL =    28.0


2. trained model, TRAINING text:      PPL =    2.62
3. trained model, held-out validation:PPL =    5.75
4. trained model, out-of-domain text: PPL =    49.0

Read the four numbers together:
  • Untrained, PPL 28.0 ≈ the 29-character vocabulary: pure guessing.
  • Trained, in-domain PPL 5.75: much better, but nowhere near 1.0.
    Real prose is genuinely uncertain. A language model reporting PPL ≈ 1
    has memorised its corpus, not learned a language — which is exactly the
    false intuition a repeated toy corpus teaches.
  • Training 2.62 vs held-out 5.75 = a 2.2× generalisation gap.
    The model fits text it has seen better than text it has not; measuring
    perplexity on training data would have flattered it.
  • Out-of-domain PPL 49.0 is worse than the untrained model's 28.0:
    on dispatch text the model is confidently wrong, which costs more than
    guessing. Perplexity measures fit BETWEEN a model and a text — it is not
    a quality score for text on its own.


## 📚 References & Further Reading

**Papers:**
- Papineni et al. (2002) — [BLEU](https://aclanthology.org/P02-1040/)
- Jelinek et al. (1977) — perplexity's origin in speech recognition; see also [The Gradient — Understanding Perplexity](https://thegradient.pub/understanding-evaluation-metrics-for-language-models/)

- Zhang et al. (2020) — [BERTScore: Evaluating Text Generation with BERT](https://arxiv.org/abs/1904.09675) *(ICLR 2020; the semantic-similarity metric named below)*

**Beyond BLEU/perplexity:** ROUGE (summarization), BERTScore (semantic similarity), and LLM-as-judge evaluations are today's standard complements.


## 📝 Summary

In **06 — Evaluating Text Generation Quality** you computed both metrics on **real text**, never on sentences written to make the demo work.

**BLEU** was scored against a real sentence from Austen'''s *Emma* — `can i do any thing for you no i thank you` — with candidates selected automatically by *measured* word overlap: the closest real sentence in the same book, a moderate-overlap one, and the least-overlapping sentence in *Moby Dick*.

| candidate | word overlap | BLEU |
|---|---|---|
| identical copy | 1.00 | 1.0000 |
| closest real sentence, same book | 0.78 | 0.5779 |
| moderate-overlap real sentence | 0.22 | 0.0406 |
| unrelated real sentence (*Moby Dick*) | 0.00 | 0.0000 |

BLEU fell monotonically with overlap, as it should. The instructive part is **how steep the penalty is**: `can i do any thing for you oh` keeps 78% of the reference'''s words and still loses 42% of the score, because BLEU counts n-grams *in order* and one edited word breaks every 4-gram covering it. Curated candidates would have hidden that.

**Perplexity** was measured four ways on a character LM trained on real *Emma* prose:

| measurement | PPL |
|---|---|
| untrained, validation text | 28.0 |
| trained, training text | 2.62 |
| trained, held-out validation | 5.75 |
| trained, out-of-domain 911 dispatch titles | 49.0 |

Real data is what makes these numbers interpretable. Untrained perplexity lands at the 29-character vocabulary size — pure guessing. In-domain perplexity settles at 5.75, **not** at 1.0: **a model reporting perplexity ≈ 1 has memorised its corpus, not learned a language**, which is exactly the false intuition a repeated toy corpus teaches. The 2.2× training-vs-held-out gap is the generalisation gap made visible. And out-of-domain perplexity (49.0) is *worse than the untrained model*: on text it never saw, the model is confidently wrong, and confident errors cost more than guessing.

Together with Unit 1'''s FID work (example 10), you now have the standard evaluation toolkit for generative models.
